# CrashLens — CarDD Instance Segmentation (YOLOv8s-seg)

This notebook mirrors the structure, configuration, and methodology of
`car_damage_detection_targeted_hardclass_experiments` (detection, bounding boxes), adapted
for **instance segmentation**: it trains and evaluates YOLOv8s-seg on the same CarDD
dataset, same 6 damage classes, same train/val/test splits, so detection and segmentation
results stay directly comparable.

Goal: train YOLOv8s-seg on CarDD_COCO (polygon annotations), compare **SGD vs AdamW**,
tune thresholds on validation, and evaluate once on the test set — same protocol as
the detection notebook, with mask metrics now also captured everywhere box metrics are.

Protocol:
1. Convert CarDD COCO polygon/RLE annotations to YOLO-seg polygon labels (train/val/test).
2. Keep the same hard-class photometric augmentation step as detection (toggle-controlled).
3. Train candidate models (SGD vs AdamW) on the training split.
4. Select the best model using validation **mask** mAP@0.5:0.95 only.
5. Tune confidence/IoU thresholds using validation only.
6. Evaluate once on the test set after all decisions are finalized.
7. Save every metric, table, and figure (box + mask) to Drive.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics pycocotools -q

## 2. Imports, Paths, and Global Settings

In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
import json
import shutil
import os
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Same CarDD source as the detection notebook, for consistency and comparability.
# CarDD_COCO holds the polygon/RLE 'segmentation' field instance segmentation needs.
# CarDD_SOD is deliberately NOT used -- it is salient-object (binary, class-agnostic)
# masks, not per-instance per-class polygons.
DATASET_ROOT = Path('/content/drive/MyDrive/CarDD_release/CarDD_release/CarDD_COCO')

# Temporary Colab datasets (same pattern as detection: build locally, copy to Drive).
ORIGINAL_YOLO_DIR = Path('/content/cardd_seg_yolo_original')
TARGET_YOLO_DIR = Path('/content/cardd_seg_yolo_hardclass_augmented')

# New, empty Drive folder. Everything this notebook produces lives under it -- no other
# Drive folder (including the detection notebook's experiment folders) is touched.
DRIVE_SEG_DIR = Path('/content/drive/MyDrive/cardd_seg')
FIGURES_DIR = DRIVE_SEG_DIR / 'figures'
BASE_RUN_DIR = Path('/content/cardd_seg_experiments')

BASE_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_SEG_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    'dent',
    'scratch',
    'crack',
    'glass shatter',
    'lamp broken',
    'tire flat'
]

HARD_CLASS_IDS = {0, 1, 2}  # dent, scratch, crack
HARD_CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(HARD_CLASS_IDS)]

# Segmentation adaptation: -seg checkpoint. Everything else matches the detection config.
BASE_MODEL = 'yolov8s-seg.pt'
SELECTED_IMGSZ = 640       # same as detection
SELECTED_EPOCHS = 150      # same as detection
BATCH_SIZE = 16
PATIENCE = 30

# Start conservatively: one augmented copy per hard-class image (same as detection).
AUGMENTED_COPIES_PER_IMAGE = 1

# Toggle: the hard-class augmentation step is photometric-only, so polygons/masks stay
# valid either way. Kept on by default to mirror detection, but flag-controlled in case
# it turns out to just add noise for a first segmentation baseline (see Section 4).
APPLY_HARDCLASS_AUGMENTATION = True

print('Drive segmentation folder:', DRIVE_SEG_DIR)
print('Figures folder:', FIGURES_DIR)
print('Hard classes:', HARD_CLASS_NAMES)
print('Hard-class augmentation enabled:', APPLY_HARDCLASS_AUGMENTATION)

## 3. Convert CarDD COCO Annotations to YOLO-SEG (Polygons)

In [ ]:
splits = {
    'train': 'train2017',
    'val': 'val2017',
    'test': 'test2017'
}

category_map = {
    1: 0,  # dent
    2: 1,  # scratch
    3: 2,  # crack
    4: 3,  # glass shatter
    5: 4,  # lamp broken
    6: 5   # tire flat
}


def _polygon_area(poly):
    """Shoelace formula on a flat [x1, y1, x2, y2, ...] polygon."""
    xs = poly[0::2]
    ys = poly[1::2]
    n = len(xs)
    if n < 3:
        return 0.0
    total = 0.0
    for i in range(n):
        j = (i + 1) % n
        total += xs[i] * ys[j] - xs[j] * ys[i]
    return abs(total) / 2.0


def _largest_polygon(polygons):
    """CarDD instances occasionally have multiple polygon rings (e.g. a damage split
    by an occluding edge). We keep the single largest-area ring per instance rather than
    writing every ring as a separate YOLO-seg instance, so one annotation stays one
    label line -- simplest and avoids inventing instance identity COCO didn't give us."""
    if len(polygons) == 1:
        return polygons[0]
    return max(polygons, key=_polygon_area)


def _polygon_from_rle(rle, img_h, img_w):
    """Fallback for RLE segmentation masks: decode to a binary mask, take the largest
    external contour, and use it as the instance polygon."""
    from pycocotools import mask as maskUtils

    if isinstance(rle['counts'], list):
        rle = maskUtils.frPyObjects(rle, img_h, img_w)
    binary_mask = maskUtils.decode(rle)
    contours, _ = cv2.findContours(
        binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    if len(largest) < 3:
        return None
    return largest.reshape(-1).astype(float).tolist()


def annotation_to_polygon(ann, img_h, img_w):
    """Returns one flat [x1, y1, x2, y2, ...] polygon in pixel coords, or None."""
    seg = ann.get('segmentation')
    if seg is None:
        return None
    if isinstance(seg, list):
        # List of polygon rings (each a flat list of coords).
        valid = [p for p in seg if len(p) >= 6]
        if not valid:
            return None
        return _largest_polygon(valid)
    if isinstance(seg, dict):
        # RLE (compressed or uncompressed counts).
        return _polygon_from_rle(seg, img_h, img_w)
    return None


def convert_coco_to_yolo_seg(dataset_root: Path, yolo_dir: Path):
    yolo_dir.mkdir(parents=True, exist_ok=True)

    conversion_report = {}

    for split, folder in splits.items():
        images_dir = yolo_dir / 'images' / split
        labels_dir = yolo_dir / 'labels' / split
        images_dir.mkdir(parents=True, exist_ok=True)
        labels_dir.mkdir(parents=True, exist_ok=True)

        ann_file = dataset_root / 'annotations' / f'instances_{folder}.json'
        with open(ann_file, 'r') as f:
            coco = json.load(f)

        images = {img['id']: img for img in coco['images']}

        annotations_by_image = defaultdict(list)
        for ann in coco['annotations']:
            annotations_by_image[ann['image_id']].append(ann)

        missing_images = []
        skipped_annotations = 0
        written_instances = 0

        for image_id, img_info in images.items():
            file_name = img_info['file_name']
            img_w = img_info['width']
            img_h = img_info['height']

            src_img = dataset_root / folder / file_name
            dst_img = images_dir / file_name

            if src_img.exists():
                if not dst_img.exists():
                    shutil.copy(src_img, dst_img)
            else:
                missing_images.append(file_name)
                continue

            label_file = labels_dir / Path(file_name).with_suffix('.txt').name
            with open(label_file, 'w') as lf:
                for ann in annotations_by_image.get(image_id, []):
                    cat_id = ann['category_id']
                    if cat_id not in category_map:
                        continue

                    polygon = annotation_to_polygon(ann, img_h, img_w)
                    if polygon is None or len(polygon) < 6:
                        skipped_annotations += 1
                        continue

                    class_id = category_map[cat_id]
                    norm_coords = []
                    for i in range(0, len(polygon), 2):
                        norm_coords.append(polygon[i] / img_w)
                        norm_coords.append(polygon[i + 1] / img_h)

                    coord_str = ' '.join(f'{v:.6f}' for v in norm_coords)
                    lf.write(f'{class_id} {coord_str}\n')
                    written_instances += 1

        conversion_report[split] = {
            'images_converted': len(images) - len(missing_images),
            'images_missing': len(missing_images),
            'instances_written': written_instances,
            'annotations_skipped_no_valid_polygon': skipped_annotations
        }

        print(f'{split}: converted {len(images) - len(missing_images)} images, '
              f'{written_instances} polygon instances, '
              f'{skipped_annotations} annotations skipped (no usable polygon)')
        if missing_images:
            print(f'{split}: missing {len(missing_images)} images')

    return conversion_report


coco_to_seg_report = convert_coco_to_yolo_seg(DATASET_ROOT, ORIGINAL_YOLO_DIR)

with open(DRIVE_SEG_DIR / 'coco_to_yoloseg_conversion_report.json', 'w') as f:
    json.dump(coco_to_seg_report, f, indent=4)

In [ ]:
# Save the converted (non-augmented) YOLO-seg dataset to Drive, under cardd_seg --
# mirrors the detection notebook's dataset backup step, kept inside our own folder.
YOLO_SEG_DRIVE_DATA = DRIVE_SEG_DIR / 'cardd_yolo_seg_data'
if YOLO_SEG_DRIVE_DATA.exists():
    shutil.rmtree(YOLO_SEG_DRIVE_DATA)
shutil.copytree(ORIGINAL_YOLO_DIR, YOLO_SEG_DRIVE_DATA)
print('Saved successfully:', YOLO_SEG_DRIVE_DATA)

## 4. Create Targeted Hard-Class Augmented Training Dataset

Same step as the detection notebook: extra augmented copies of **training images that
contain at least one hard class** (dent, scratch, crack).

- Training split is augmented.
- Validation split is copied unchanged.
- Test split is copied unchanged.
- Polygons/masks stay valid because augmentations are photometric only (no geometric
  transform touches pixel coordinates).

Controlled by `APPLY_HARDCLASS_AUGMENTATION` (Section 2). If this turns out to just add
noise for a first segmentation baseline, flip it to `False` and rerun this cell -- the
rest of the notebook doesn't need to change, it always trains on `TARGET_YOLO_DIR`.

In [ ]:
def read_label_classes(label_path: Path):
    classes = set()
    if not label_path.exists():
        return classes
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                classes.add(int(parts[0]))
    return classes

def photometric_hardclass_augment(image):
    # Photometric transforms only: labels (polygons) remain unchanged.
    aug = image.copy()

    # Brightness/contrast
    alpha = random.uniform(0.90, 1.20)
    beta = random.randint(-12, 18)
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)

    # Mild sharpening
    if random.random() < 0.7:
        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])
        aug = cv2.filter2D(aug, -1, kernel)

    # Mild Gaussian noise
    if random.random() < 0.5:
        noise = np.random.normal(0, 5, aug.shape).astype(np.int16)
        aug = np.clip(aug.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    return aug

def build_hardclass_augmented_dataset(original_yolo_dir: Path, target_yolo_dir: Path, hard_class_ids, copies_per_image: int = 1):
    if target_yolo_dir.exists():
        shutil.rmtree(target_yolo_dir)

    shutil.copytree(original_yolo_dir, target_yolo_dir)

    train_images_dir = target_yolo_dir / 'images' / 'train'
    train_labels_dir = target_yolo_dir / 'labels' / 'train'

    image_extensions = {'.jpg', '.jpeg', '.png'}
    train_images = [p for p in train_images_dir.iterdir() if p.suffix.lower() in image_extensions]

    selected_for_augmentation = []
    added_images = 0

    for image_path in train_images:
        label_path = train_labels_dir / image_path.with_suffix('.txt').name
        classes = read_label_classes(label_path)

        if classes & hard_class_ids:
            selected_for_augmentation.append(image_path)
            image = cv2.imread(str(image_path))
            if image is None:
                continue

            for copy_idx in range(1, copies_per_image + 1):
                aug_image = photometric_hardclass_augment(image)
                aug_stem = f'{image_path.stem}_hardaug{copy_idx}'
                aug_image_path = train_images_dir / f'{aug_stem}{image_path.suffix}'
                aug_label_path = train_labels_dir / f'{aug_stem}.txt'

                cv2.imwrite(str(aug_image_path), aug_image)
                shutil.copy(label_path, aug_label_path)
                added_images += 1

    return {
        'hard_classes': [CLASS_NAMES[i] for i in sorted(hard_class_ids)],
        'copies_per_image': copies_per_image,
        'selected_train_images': len(selected_for_augmentation),
        'added_augmented_images': added_images
    }

if APPLY_HARDCLASS_AUGMENTATION:
    augmentation_summary = build_hardclass_augmented_dataset(
        original_yolo_dir=ORIGINAL_YOLO_DIR,
        target_yolo_dir=TARGET_YOLO_DIR,
        hard_class_ids=HARD_CLASS_IDS,
        copies_per_image=AUGMENTED_COPIES_PER_IMAGE
    )
else:
    if TARGET_YOLO_DIR.exists():
        shutil.rmtree(TARGET_YOLO_DIR)
    shutil.copytree(ORIGINAL_YOLO_DIR, TARGET_YOLO_DIR)
    augmentation_summary = {
        'hard_classes': [CLASS_NAMES[i] for i in sorted(HARD_CLASS_IDS)],
        'copies_per_image': 0,
        'selected_train_images': 0,
        'added_augmented_images': 0,
        'note': 'APPLY_HARDCLASS_AUGMENTATION was False -- TARGET_YOLO_DIR is an unmodified copy of ORIGINAL_YOLO_DIR.'
    }

print(json.dumps(augmentation_summary, indent=4))

with open(DRIVE_SEG_DIR / 'hardclass_augmentation_summary.json', 'w') as f:
    json.dump(augmentation_summary, f, indent=4)

## 5. Create `data.yaml` Files

In [ ]:
def write_data_yaml(yolo_dir: Path, yaml_path: Path):
    yaml_lines = [
        f'path: {yolo_dir}',
        'train: images/train',
        'val: images/val',
        'test: images/test',
        '',
        'nc: 6',
        '',
        'names:',
        '  0: dent',
        '  1: scratch',
        '  2: crack',
        '  3: glass shatter',
        '  4: lamp broken',
        '  5: tire flat'
    ]
    yaml_content = '\n'.join(yaml_lines) + '\n'
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    print(f'Created: {yaml_path}')
    print(open(yaml_path).read())

original_data_yaml_path = ORIGINAL_YOLO_DIR / 'data.yaml'
target_data_yaml_path = TARGET_YOLO_DIR / 'data.yaml'

write_data_yaml(ORIGINAL_YOLO_DIR, original_data_yaml_path)
write_data_yaml(TARGET_YOLO_DIR, target_data_yaml_path)

## 6. Dataset Verification, Leakage Check, and Polygon-Format Check

In [ ]:
image_extensions = {'.jpg', '.jpeg', '.png'}

def verify_yolo_dataset(yolo_dir: Path, name: str):
    print(f'\nDataset: {name}')
    verification = {}

    for split in ['train', 'val', 'test']:
        images_dir = yolo_dir / 'images' / split
        labels_dir = yolo_dir / 'labels' / split

        image_files = [p for p in images_dir.iterdir() if p.suffix.lower() in image_extensions]
        label_files = list(labels_dir.glob('*.txt'))

        verification[split] = {'images': len(image_files), 'labels': len(label_files)}
        print(f'{split}: images={len(image_files)}, labels={len(label_files)}')

    train_files = {p.name for p in (yolo_dir / 'images/train').iterdir() if p.suffix.lower() in image_extensions}
    val_files = {p.name for p in (yolo_dir / 'images/val').iterdir() if p.suffix.lower() in image_extensions}
    test_files = {p.name for p in (yolo_dir / 'images/test').iterdir() if p.suffix.lower() in image_extensions}

    overlaps = {
        'train_val_overlap': len(train_files & val_files),
        'train_test_overlap': len(train_files & test_files),
        'val_test_overlap': len(val_files & test_files)
    }
    print('Train-Val overlap:', overlaps['train_val_overlap'])
    print('Train-Test overlap:', overlaps['train_test_overlap'])
    print('Val-Test overlap:', overlaps['val_test_overlap'])

    verification['overlaps'] = overlaps
    return verification


def verify_polygon_format(labels_dir: Path, sample_size: int = 50):
    """Segmentation adaptation: confirm labels are YOLO-seg polygon lines
    ('class x1 y1 x2 y2 ... xn yn', odd token count, >=7 tokens), NOT the 5-token
    bbox format ('class x_center y_center w h') the detection notebook writes."""
    label_files = list(labels_dir.glob('*.txt'))[:sample_size]
    checked_lines = 0
    bbox_shaped_lines = 0
    min_tokens = None
    max_tokens = None

    for label_file in label_files:
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                n_tokens = len(parts)
                checked_lines += 1
                min_tokens = n_tokens if min_tokens is None else min(min_tokens, n_tokens)
                max_tokens = n_tokens if max_tokens is None else max(max_tokens, n_tokens)
                if n_tokens == 5:
                    bbox_shaped_lines += 1
                assert n_tokens >= 7, f'{label_file.name}: line has only {n_tokens} tokens, too few for a polygon'
                assert (n_tokens - 1) % 2 == 0, f'{label_file.name}: {n_tokens} tokens is not class_id + an even number of coords'

    result = {
        'files_sampled': len(label_files),
        'lines_checked': checked_lines,
        'bbox_shaped_lines_found': bbox_shaped_lines,
        'min_tokens_per_line': min_tokens,
        'max_tokens_per_line': max_tokens
    }
    assert bbox_shaped_lines == 0, 'Found bbox-shaped (5-token) label lines -- conversion did not write polygons.'
    print(f'Polygon-format check on {labels_dir}: {result}')
    return result


original_verification = verify_yolo_dataset(ORIGINAL_YOLO_DIR, 'original')
target_verification = verify_yolo_dataset(TARGET_YOLO_DIR, 'hard-class augmented')

polygon_format_check = verify_polygon_format(ORIGINAL_YOLO_DIR / 'labels/train')

with open(DRIVE_SEG_DIR / 'dataset_verification.json', 'w') as f:
    json.dump({
        'original': original_verification,
        'hardclass_augmented': target_verification,
        'polygon_format_check': polygon_format_check
    }, f, indent=4)

## 7. Class Distribution Counts (Original vs Augmented)

In [ ]:
def count_yolo_labels(labels_dir: Path, class_names):
    counts = Counter()
    for label_file in labels_dir.glob('*.txt'):
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_id = int(parts[0])
                    counts[class_names[class_id]] += 1
    return counts

original_train_counts = count_yolo_labels(ORIGINAL_YOLO_DIR / 'labels/train', CLASS_NAMES)
target_train_counts = count_yolo_labels(TARGET_YOLO_DIR / 'labels/train', CLASS_NAMES)

class_count_df = pd.DataFrame({
    'class': CLASS_NAMES,
    'original_train_instances': [original_train_counts.get(c, 0) for c in CLASS_NAMES],
    'augmented_train_instances': [target_train_counts.get(c, 0) for c in CLASS_NAMES],
})
class_count_df['increase'] = class_count_df['augmented_train_instances'] - class_count_df['original_train_instances']

display(class_count_df)
class_count_df.to_csv(DRIVE_SEG_DIR / 'class_distribution_original_vs_augmented.csv', index=False)

## 8. Experiment Configuration

In [ ]:
EXPERIMENTS = [
    {
        'phase': 'cardd_seg_optimizer_comparison',
        'name': f'yolov8s_seg_img{SELECTED_IMGSZ}_hardaug_sgd_ep{SELECTED_EPOCHS}',
        'dataset_variant': 'hardclass_augmented',
        'model': BASE_MODEL,
        'imgsz': SELECTED_IMGSZ,
        'epochs': SELECTED_EPOCHS,
        'batch': BATCH_SIZE,
        'patience': PATIENCE,
        'optimizer': 'SGD',
        'lr0': 0.01,
        'momentum': 0.937,
        'weight_decay': 0.0005
    },
    {
        'phase': 'cardd_seg_optimizer_comparison',
        'name': f'yolov8s_seg_img{SELECTED_IMGSZ}_hardaug_adamw_ep{SELECTED_EPOCHS}',
        'dataset_variant': 'hardclass_augmented',
        'model': BASE_MODEL,
        'imgsz': SELECTED_IMGSZ,
        'epochs': SELECTED_EPOCHS,
        'batch': BATCH_SIZE,
        'patience': PATIENCE,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'weight_decay': 0.01
    }
]

with open(DRIVE_SEG_DIR / 'experiments_config.json', 'w') as f:
    json.dump(EXPERIMENTS, f, indent=4)

print(json.dumps(EXPERIMENTS, indent=4))

## 9. Training and Validation Loop

In [ ]:
from ultralytics import YOLO

# Every PNG/JPG Ultralytics auto-generates per run (box AND mask versions) that we copy
# out of each run's local directory into Drive's flat figures/ folder, prefixed by run
# name, so every figure survives even though the full run dirs are also copied to Drive.
FIGURE_FILENAME_PATTERNS = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'labels.jpg',
    'labels_correlogram.jpg',
    'BoxP_curve.png', 'BoxR_curve.png', 'BoxPR_curve.png', 'BoxF1_curve.png',
    'MaskP_curve.png', 'MaskR_curve.png', 'MaskPR_curve.png', 'MaskF1_curve.png',
    'val_batch*_labels.jpg', 'val_batch*_pred.jpg'
]

def copy_run_figures_to_drive(run_source_dir: Path, figures_dir: Path, run_label: str):
    """Globs the known Ultralytics output figures out of a run directory (recursively,
    since train/ and val/ subfolders both produce figures) and copies each into the flat
    Drive figures/ folder, prefixed with run_label to avoid collisions across runs."""
    figures_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for pattern in FIGURE_FILENAME_PATTERNS:
        for src_path in run_source_dir.rglob(pattern):
            dst_name = f'{run_label}__{src_path.name}'
            dst_path = figures_dir / dst_name
            shutil.copy(src_path, dst_path)
            copied.append(dst_name)
    return copied


def run_experiment(exp, data_yaml_path: Path):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name = f"{exp['name']}_{timestamp}"
    local_run = BASE_RUN_DIR / run_name
    drive_run = DRIVE_SEG_DIR / run_name
    local_run.mkdir(parents=True, exist_ok=True)

    with open(local_run / 'experiment_config.json', 'w') as f:
        json.dump(exp, f, indent=4)

    print(f"\nStarting experiment: {run_name}")

    model = YOLO(exp['model'])

    train_kwargs = {
        'data': str(data_yaml_path),
        'imgsz': exp['imgsz'],
        'epochs': exp['epochs'],
        'batch': exp['batch'],
        'patience': exp['patience'],
        'project': str(local_run / 'training'),
        'name': 'train',
        'exist_ok': True,
        'seed': SEED,
        'deterministic': True,
        'optimizer': exp['optimizer'],
        'lr0': exp['lr0'],
        'weight_decay': exp['weight_decay']
    }

    if 'momentum' in exp:
        train_kwargs['momentum'] = exp['momentum']

    model.train(**train_kwargs)

    best_weights = local_run / 'training' / 'train' / 'weights' / 'best.pt'
    if not best_weights.exists():
        raise FileNotFoundError(f'Best weights not found: {best_weights}')

    model = YOLO(str(best_weights))

    val_result = model.val(
        data=str(data_yaml_path),
        split='val',
        imgsz=exp['imgsz'],
        project=str(local_run / 'validation'),
        name='val',
        exist_ok=True
    )

    summary = {
        'run_name': run_name,
        'source': 'cardd_segmentation_experiment',
        'weights_path': str(best_weights),
        **exp,
        # Box metrics (kept in every table alongside mask metrics, per the task spec).
        'val_box_mAP50': float(val_result.box.map50),
        'val_box_mAP50_95': float(val_result.box.map),
        'val_box_precision': float(val_result.box.mp),
        'val_box_recall': float(val_result.box.mr),
        # Mask metrics -- the primary model-selection metric for a segmentation model.
        'val_seg_mAP50': float(val_result.seg.map50),
        'val_seg_mAP50_95': float(val_result.seg.map),
        'val_seg_precision': float(val_result.seg.mp),
        'val_seg_recall': float(val_result.seg.mr)
    }

    for class_name, value in zip(CLASS_NAMES, val_result.box.maps):
        summary[f'val_box_map50_95_{class_name}'] = float(value)
    for class_name, value in zip(CLASS_NAMES, val_result.seg.maps):
        summary[f'val_seg_map50_95_{class_name}'] = float(value)

    with open(local_run / 'summary.json', 'w') as f:
        json.dump(summary, f, indent=4)

    if drive_run.exists():
        shutil.rmtree(drive_run)

    shutil.copytree(local_run, drive_run)
    summary['weights_path'] = str(drive_run / 'training' / 'train' / 'weights' / 'best.pt')

    copied_figures = copy_run_figures_to_drive(drive_run, FIGURES_DIR, run_name)
    print(f'Copied {len(copied_figures)} figure(s) to {FIGURES_DIR}')
    print(f'Saved to Drive: {drive_run}')
    return summary

all_results = []

for exp in EXPERIMENTS:
    summary = run_experiment(exp, target_data_yaml_path)
    all_results.append(summary)

new_results_df = pd.DataFrame(all_results).sort_values(
    by='val_seg_mAP50_95',
    ascending=False
).reset_index(drop=True)

new_results_df.to_csv(DRIVE_SEG_DIR / 'seg_experiment_comparison.csv', index=False)
display(new_results_df)

## 10. Optional: Add Previous Best Baseline as a Reference Row

In [ ]:
# No prior CarDD segmentation run exists yet -- this points at where one would live if
# produced later, and gracefully skips (same pattern as the detection notebook, which
# reads its own separate CrashLens_ML_Experiments reference folder). Comparing a
# detection-only run here wouldn't be meaningful (box-only vs box+mask), so this is
# intentionally scoped to a segmentation-specific reference file only.
PREVIOUS_EXPERIMENTS_DIR = Path('/content/drive/MyDrive/CrashLens_ML_Segmentation_Experiments')
PREVIOUS_COMPARISON_CSV = PREVIOUS_EXPERIMENTS_DIR / 'seg_experiment_comparison.csv'

reference_rows = []

if PREVIOUS_COMPARISON_CSV.exists():
    prev_df = pd.read_csv(PREVIOUS_COMPARISON_CSV)
    if 'val_seg_mAP50_95' in prev_df.columns and len(prev_df) > 0:
        prev_best = prev_df.sort_values(by='val_seg_mAP50_95', ascending=False).iloc[0].to_dict()
        prev_best['source'] = 'previous_segmentation_reference'

        prev_run_name = prev_best.get('run_name', '')
        prev_weights = PREVIOUS_EXPERIMENTS_DIR / str(prev_run_name) / 'training' / 'train' / 'weights' / 'best.pt'
        prev_best['weights_path'] = str(prev_weights) if prev_weights.exists() else ''

        reference_rows.append(prev_best)
        print('Loaded previous best reference:', prev_run_name)
    else:
        print('Previous comparison file found, but val_seg_mAP50_95 column is missing or file is empty.')
else:
    print('Previous segmentation comparison CSV not found. Skipping reference row.')

combined_results_df = pd.concat(
    [pd.DataFrame(reference_rows), new_results_df],
    ignore_index=True,
    sort=False
)

combined_results_df = combined_results_df.sort_values(
    by='val_seg_mAP50_95',
    ascending=False
).reset_index(drop=True)

combined_results_df.to_csv(DRIVE_SEG_DIR / 'combined_comparison_with_previous_reference.csv', index=False)
display(combined_results_df)

## 11. Select Best Model Based on Validation (Mask mAP@0.5:0.95)

In [ ]:
best_experiment = combined_results_df.iloc[0]
best_run_name = best_experiment['run_name']
selected_weights = Path(best_experiment['weights_path'])
selected_imgsz = int(best_experiment['imgsz'])

if not selected_weights.exists():
    raise FileNotFoundError(
        f'Selected weights do not exist: {selected_weights}. '
        'If the previous reference was selected but weights are missing, select one of the new runs manually.'
    )

print('Selected best validation model (by MASK mAP@0.5:0.95):')
display(best_experiment)
print('Selected weights:', selected_weights)

## 12. Per-Class Validation Metrics for Selected Model (Mask)

In [ ]:
per_class_cols = [col for col in combined_results_df.columns if str(col).startswith('val_seg_map50_95_')]

per_class_df = best_experiment[per_class_cols].reset_index()
per_class_df.columns = ['metric', 'val_seg_mAP50_95']
per_class_df['class'] = per_class_df['metric'].str.replace('val_seg_map50_95_', '', regex=False)
per_class_df = per_class_df[['class', 'val_seg_mAP50_95']]

display(per_class_df)
per_class_df.to_csv(DRIVE_SEG_DIR / 'selected_model_per_class_validation_metrics.csv', index=False)

In [ ]:
per_class_df.set_index('class')['val_seg_mAP50_95'].plot(kind='bar')
plt.title('Selected Model — Validation Per-Class MASK mAP@0.5:0.95')
plt.ylabel('Mask mAP@0.5:0.95')
plt.xlabel('Damage Class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(DRIVE_SEG_DIR / 'selected_model_per_class_validation_map.png', dpi=200)
plt.savefig(FIGURES_DIR / 'selected_model_per_class_validation_mask_map.png', dpi=200)
plt.show()

## 13. Threshold and IoU Tuning on Validation

In [ ]:
THRESHOLD_GRID = [
    {'conf': 0.15, 'iou': 0.50},
    {'conf': 0.15, 'iou': 0.70},
    {'conf': 0.25, 'iou': 0.50},
    {'conf': 0.25, 'iou': 0.70},
    {'conf': 0.35, 'iou': 0.50},
    {'conf': 0.35, 'iou': 0.70},
]

threshold_dir = DRIVE_SEG_DIR / 'threshold_tuning'
threshold_dir.mkdir(parents=True, exist_ok=True)

threshold_results = []
best_model = YOLO(str(selected_weights))

for item in THRESHOLD_GRID:
    conf = item['conf']
    iou = item['iou']
    run_label = f"conf{conf}_iou{iou}"

    val_threshold_result = best_model.val(
        data=str(target_data_yaml_path),
        split='val',
        imgsz=selected_imgsz,
        conf=conf,
        iou=iou,
        project=str(threshold_dir),
        name=run_label,
        exist_ok=True
    )

    # Precision/recall/F1 stay box-based (same as the detection notebook's methodology);
    # both box and mask mAP are recorded alongside them per the task's metric spec.
    precision = float(val_threshold_result.box.mp)
    recall = float(val_threshold_result.box.mr)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    threshold_results.append({
        'selected_run_name': best_run_name,
        'conf': conf,
        'iou': iou,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'box_mAP50': float(val_threshold_result.box.map50),
        'box_mAP50_95': float(val_threshold_result.box.map),
        'mask_mAP50': float(val_threshold_result.seg.map50),
        'mask_mAP50_95': float(val_threshold_result.seg.map)
    })

    copy_run_figures_to_drive(threshold_dir / run_label, FIGURES_DIR, f'threshold_{run_label}')

threshold_df = pd.DataFrame(threshold_results).sort_values(by='f1', ascending=False).reset_index(drop=True)
threshold_df.to_csv(threshold_dir / 'threshold_comparison.csv', index=False)
display(threshold_df)

## 14. Formal Test Evaluation

In [ ]:
best_threshold = threshold_df.iloc[0]

test_dir = DRIVE_SEG_DIR / 'final_test'
test_dir.mkdir(parents=True, exist_ok=True)

test_result = best_model.val(
    data=str(target_data_yaml_path),
    split='test',
    imgsz=selected_imgsz,
    conf=float(best_threshold['conf']),
    iou=float(best_threshold['iou']),
    project=str(test_dir),
    name='test',
    exist_ok=True
)

test_summary = {
    'selected_model': best_run_name,
    'selected_weights': str(selected_weights),
    'selected_imgsz': selected_imgsz,
    'selected_conf': float(best_threshold['conf']),
    'selected_iou': float(best_threshold['iou']),
    'test_box_mAP50': float(test_result.box.map50),
    'test_box_mAP50_95': float(test_result.box.map),
    'test_box_precision': float(test_result.box.mp),
    'test_box_recall': float(test_result.box.mr),
    'test_mask_mAP50': float(test_result.seg.map50),
    'test_mask_mAP50_95': float(test_result.seg.map),
    'test_mask_precision': float(test_result.seg.mp),
    'test_mask_recall': float(test_result.seg.mr)
}

for class_name, value in zip(CLASS_NAMES, test_result.box.maps):
    test_summary[f'test_box_map50_95_{class_name}'] = float(value)
for class_name, value in zip(CLASS_NAMES, test_result.seg.maps):
    test_summary[f'test_mask_map50_95_{class_name}'] = float(value)

with open(test_dir / 'test_summary.json', 'w') as f:
    json.dump(test_summary, f, indent=4)

copy_run_figures_to_drive(test_dir / 'test', FIGURES_DIR, 'final_test')

test_summary

## 15. Prediction Samples for Visual Inspection

In [ ]:
prediction_dir = DRIVE_SEG_DIR / 'predictions'

# save=True renders mask overlays automatically for a -seg model (not just boxes).
best_model.predict(
    source=str(TARGET_YOLO_DIR / 'images/test'),
    imgsz=selected_imgsz,
    conf=float(best_threshold['conf']),
    iou=float(best_threshold['iou']),
    save=True,
    project=str(prediction_dir),
    name='test_predictions',
    exist_ok=True
)

print('Predictions (with mask overlays) saved to:', prediction_dir)

## 16. Final Output Structure

Expected Drive structure (everything lives under `cardd_seg/` -- no other Drive folder
is touched):

```text
cardd_seg/
├── coco_to_yoloseg_conversion_report.json
├── cardd_yolo_seg_data/                  (converted, non-augmented YOLO-seg dataset)
├── hardclass_augmentation_summary.json
├── dataset_verification.json             (incl. leakage check + polygon-format check)
├── class_distribution_original_vs_augmented.csv
├── experiments_config.json
├── seg_experiment_comparison.csv
├── combined_comparison_with_previous_reference.csv
├── selected_model_per_class_validation_metrics.csv
├── selected_model_per_class_validation_map.png
├── figures/                               (every results/PR/F1/confusion-matrix/labels/
│                                          val_batch*_pred figure, box AND mask versions,
│                                          copied out of every run, prefixed by run name)
├── threshold_tuning/
│   └── threshold_comparison.csv
├── final_test/
│   └── test/
├── predictions/
│   └── test_predictions/
└── <each training run>/
    ├── experiment_config.json
    ├── summary.json                       (box + mask metrics)
    ├── training/
    └── validation/
```